In [1]:
import numpy as np
import matplotlib.pyplot as plt
from tanalysis import improcess, stitching
from skimage import exposure


dirname = r"C:\Users\pcanaleta\Documents\Cellpose_segmentation\EXP.HD6.Chips\EXP.HD6.1.1.MatekChips_CXCL10\24h\Originals\24h_CXCL10_Conc10_z5_t8h.lif"

In [2]:
imgs, dim, names, info = improcess.imread(dirname, channel=0, tiles=True)
print(imgs[0].shape)

Reading submitted files: 100%|████████████████████████████████████████| 1/1 [00:31<00:00, 31.74s/it]


All files read!
(97, 6, 14, 1024, 1024)


In [3]:
positions = info['mosaic_position']
trans = stitching.translationComputation(imgs, positions)
print(trans)

100%|██████████| 11/11 [07:14<00:00, 39.51s/it]

[[1104, -94, 304, 15], [299, 0, 301, -43], [857, -13, 877, 27], [640, 45, 849, 8], [311, 0, 966, -96], [934, 51, 1036, -81], [971, 31, 916, 14], [703, 65, 884, 42], [854, -16, 899, 8], [1091, -70, 273, 33], [976, 6, 848, 47], [1131, -191, 1007, 11], [1033, -37, 859, 31], [315, -10, 937, 30], [620, 90, 790, 44], [602, 62, 153, 69], [854, -83, 201, 44], [924, -137, 295, 72], [692, 0, 880, 6], [874, 112, 232, 12], [805, 180, 250, 54], [816, 16, 704, 4], [831, 34, 731, -22], [955, 35, 824, 86], [885, 2, 939, -36], [796, -12, 285, 17], [1105, -85, 307, 24], [812, -65, 168, 95], [855, -29, 209, 111], [698, 0, 931, 20], [992, -14, 954, -35], [981, -86, 981, -22], [792, -38, 843, -47], [901, 120, 960, -43], [987, -43, 729, 58], [729, 5, 890, 16], [974, -2, 778, 118], [1020, -16, 782, 91], [828, 0, 891, 20], [846, -10, 283, -22], [909, 27, 331, 82], [803, 28, 211, 87], [934, 63, 322, -59], [996, 8, 711, 0], [1054, -34, 884, 27], [1060, -71, 747, 1], [766, -15, 1004, -22], [827, 39, 770, 28], [8

In [5]:
atrans = np.asarray(trans)
print(atrans)
print(int(np.median(atrans[:,0])), int(np.average(atrans[:,1])), int(np.median(atrans[:,2])), int(np.average(atrans[:,3])))

[[1104  -94  304   15]
 [ 299    0  301  -43]
 [ 857  -13  877   27]
 [ 640   45  849    8]
 [ 311    0  966  -96]
 [ 934   51 1036  -81]
 [ 971   31  916   14]
 [ 703   65  884   42]
 [ 854  -16  899    8]
 [1091  -70  273   33]
 [ 976    6  848   47]
 [1131 -191 1007   11]
 [1033  -37  859   31]
 [ 315  -10  937   30]
 [ 620   90  790   44]
 [ 602   62  153   69]
 [ 854  -83  201   44]
 [ 924 -137  295   72]
 [ 692    0  880    6]
 [ 874  112  232   12]
 [ 805  180  250   54]
 [ 816   16  704    4]
 [ 831   34  731  -22]
 [ 955   35  824   86]
 [ 885    2  939  -36]
 [ 796  -12  285   17]
 [1105  -85  307   24]
 [ 812  -65  168   95]
 [ 855  -29  209  111]
 [ 698    0  931   20]
 [ 992  -14  954  -35]
 [ 981  -86  981  -22]
 [ 792  -38  843  -47]
 [ 901  120  960  -43]
 [ 987  -43  729   58]
 [ 729    5  890   16]
 [ 974   -2  778  118]
 [1020  -16  782   91]
 [ 828    0  891   20]
 [ 846  -10  283  -22]
 [ 909   27  331   82]
 [ 803   28  211   87]
 [ 934   63  322  -59]
 [ 996    8

In [ ]:
timestamp = imgs[0][90]
positions = info['mosaic_position']
grid = {}
eq_grid = {}
nrow = 0
ncol = 0
z = 5

for pos, tile in zip(positions, timestamp):
    #Recorder of number of rows and cols
    if pos[1]+1>nrow:
        nrow=pos[1]+1
    if pos[0]+1>ncol:
        ncol=pos[0]+1
    #Assigning to the dictionary the corresponding image to each combination of row, col
    eq_tile = exposure.equalize_adapthist(tile, clip_limit=0.75)
    grid[f'{pos[1]}{pos[0]}']=tile
    eq_grid[f'{pos[1]}{pos[0]}'] = eq_tile

fig, ax = plt.subplots(2,3)
for row in range(0,2):
    for col in range(0,3):
        ax[row, col].imshow(grid[f'{row}{col}'][z], cmap='gray')

fig2, ax2 = plt.subplots(2,3)
for row in range(0,2):
    for col in range(0,3):
        ax2[row, col].imshow(eq_grid[f'{row}{col}'][z], cmap='gray')

In [ ]:
translations = [] #Saving vertical an horizontal translations for each image
nccv = -np.inf
ncch = -np.inf
Tvrow = 0
Tvcol = []
Throw = []
Thcol = 0
n=20
for row in range(0, nrow):
    for col in range(0, ncol):
        im = grid[f'{row}{col}'][z]
        H,W = im.shape
        if row!=0:
            im2 = grid[f'{row-1}{col}'][z]
            nccv_, Tvrow_, Tvcol_ = stitching.pciam(im2, im, n)
            print(f'Tv:{nccv_, Tvrow_, Tvcol_}')
            if abs(Tvcol_)<int(W/5):
                Tvcol.append(Tvcol_)
            if nccv<nccv_:
                nccv = nccv_
                Tvrow = Tvrow_
        if col!=0:
            im2 = grid[f'{row}{col-1}'][z]
            ncch_, Throw_, Thcol_ = stitching.pciam(im2, im, n)
            print(f'Th:{ncch_, Throw_, Thcol_}')
            if abs(Throw_)<int(H/5):
                Throw.append(Throw_)
            if ncch<ncch_:
                ncch = ncch_
                Thcol = Thcol_

if Throw == []:
    Throw = [0]
if Tvcol == []:
    Tvcol = [0]
Tv = [int(Tvrow), int(np.average(Tvcol))]
Th = [int(np.average(Throw)), int(Thcol)]

rr = Th[0]
rc = Tv[1]
drow = Th[1]-rr
dcol = Tv[0]-rc
print(rr, drow, rc, dcol)

In [ ]:
abs_translations = {}
minr = 0
minc = 0
for row in np.arange(nrow):
    for col in np.arange(ncol):
        abs_translations[f'{row}{col}'] = [int(row*(drow+rr)+col*rr), int(row*rc+col*(dcol+rc))]
        minr_ = abs_translations[f'{row}{col}'][0]
        minc_ = abs_translations[f'{row}{col}'][1]
        if minr_<minr:
            minr=minr_
        if minc_<minc:
            minc=minc_
print(minr, minc)
print(abs_translations)

In [ ]:
H,W = np.shape(grid['00'][0])
Hmax,Wmax = abs_translations[f'{nrow-1}{ncol-1}']
rerr = abs(minr)
cerr = abs(minc)
result = np.zeros((Hmax+H+2*rerr, Wmax+W+2*cerr))
print(result.shape)
z=7

for trans in abs_translations:
    srow = abs_translations[trans][0] + rerr
    scol = abs_translations[trans][1] + cerr
    erow = srow+H
    ecol = scol+W
    result[srow:erow,scol:ecol] = result[srow:erow,scol:ecol]+grid[trans][z]


plt.figure(figsize=(18,32))
plt.imshow(result, cmap='gray')
plt.show()